# EEG ERP analysis with MEGaNorm

Use `Dataset` to describe your recordings and events, `Config` to define the analysis, and `Pipeline` to process the participants. The result is a `FeatureDataset`: one participant per row, with ERP measurements and separate quality-control reports.

**Run All requires no data download.** This example creates a small synthetic cohort and runs the same local API you will use for real EEG recordings. Section 6 shows how to replace the demonstration data and preprocessing settings with your study settings.

Component windows and sensor choices below are illustrative. They are not universal ERP definitions.

## 1. Import the local API

Select a Python 3.12 or newer notebook kernel with MEGaNorm, its dependencies and Matplotlib installed.

In [ ]:
from pathlib import Path
import json
import tempfile

import mne
import numpy as np
import pandas as pd
from IPython.display import display
from meganorm.API import Config, Dataset, Pipeline, ERPSettings, ERPPreprocessing, ERPEventSpec
from meganorm.utils.erp import make_demo_recording

mne.set_log_level("WARNING")

## 2. Create example recordings

This cell writes eight synthetic EEG recordings into a new temporary folder. Each has 20 `standard` and 20 `target` trials. The target response at Pz peaks near 300 ms with an amplitude near 6 µV.

Each participant has a folder containing a recording whose filename includes the task name, `oddball`. This is the layout the local dataset discovery expects. For your own study, use unique participant folder names and one intended recording per participant.

The temporary folder also holds the output for this demonstration. Its location is printed below; keep it if you want to inspect the files after closing the notebook.

In [ ]:
demo_dir = Path(tempfile.mkdtemp(prefix="meganorm_erp_"))
data_root = demo_dir / "recordings"
participant_ids = [f"sub-{i:03d}" for i in range(1, 9)]

for i, participant_id in enumerate(participant_ids):
    raw, events = make_demo_recording(seed=42 + i)
    raw.set_annotations(mne.annotations_from_events(
        events, sfreq=raw.info["sfreq"],
        event_desc={1: "standard", 2: "target"}, first_samp=raw.first_samp,
    ))
    folder = data_root / participant_id
    folder.mkdir(parents=True)
    raw.save(folder / f"{participant_id}_oddball_raw.fif", overwrite=False, verbose=False)

print(f"Example recordings and results: {demo_dir}")

## 3. Describe the dataset and events

`root` contains the participant folders. `task` matches text in the recording filenames; `extension` selects the file format.

The event map connects your analysis condition names on the **left** to the exact annotation descriptions in the recordings on the **right**. Here they happen to be identical. The component definitions in the next section use the names on the left.

In [ ]:
dataset = Dataset(
    name="demo_oddball", root=data_root, task="oddball", extension=".fif",
    erp_events=ERPEventSpec(
        mode="annotations",
        annotation_map={"standard": "standard", "target": "target"},
    ),
)

# Inspect the discovered participants before processing.
display(dataset.discover()[["participant_id", "recording_paths"]])

## 4. Define the ERP measurements and preprocessing

Set `analysis_type="erp"` and `which_sensor="eeg"`. ERP cleaning is controlled by `erp.preprocessing`.

Each named component specifies its conditions, individual channels, time window in **seconds**, polarity and measurements. `pos` selects the largest positive sample, `neg` the most negative, and `abs` the largest absolute sample while retaining its sign. Peak latency refers to that same sample. The window mean uses all samples within the window. All components share epoch and baseline settings.

**The preprocessing settings below are for clean synthetic data only.** ICA, filtering and re-referencing are disabled so we can check the known injected amplitudes. Baseline correction and threshold rejection remain enabled. Use the real-data settings in Section 6 for your recordings.

To fit ICA independently of the ERP analysis filter, set these fields in
`ERPPreprocessing` (the recording must have a sample rate above 200 Hz):

```python
ERPPreprocessing(
    l_freq=0.1, h_freq=30, resample_sfreq=250,
    ica_l_freq=1, ica_h_freq=100, ica_resample_sfreq=250,
)
```

This fits ICA on a 1–100 Hz copy and applies the cleaning to the 0.1–30 Hz
ERP data. Event timing follows the analysis sample rate. Omitting the two new
settings retains the previous fit filter and original fit sample rate.


In [ ]:
erp = ERPSettings(
    preprocessing=ERPPreprocessing(
        tmin=-0.2, tmax=0.8, baseline=(-0.2, 0.0),
        ica_mode="none", reference=None, l_freq=None, h_freq=None,
        min_trials=10, reject={"eeg": 150e-6},  # volts: 150 µV peak-to-peak
    ),
    components={
        "N100": {
            "conditions": ["standard", "target"], "channels": ["Cz", "Pz"],
            "window": [0.08, 0.14], "polarity": "neg",
        },
        "late_positive": {
            "conditions": ["standard", "target"], "channels": ["Pz"],
            "window": [0.2, 0.4], "polarity": "pos",
            "metrics": ["peak_amplitude", "peak_latency", "mean_amplitude"],
        },
    },
)
config = Config(analysis_type="erp", which_sensor="eeg", erp=erp)

## 5. Run the cohort and inspect the results

The pipeline discovers and processes participants, writes results, and combines usable rows automatically. No participant loop or manual CSV concatenation is needed.

`on_error="continue"` records participant failures and continues with the others. Use `"raise"` when you want to stop at the first failure. Review the processing report in either case.

In [ ]:
result = Pipeline(
    config=config, output_dir=demo_dir / "results", on_error="continue",
).run(dataset)

# Use feature_names to select only IDPs, excluding any participant metadata.
table = result.to_dataframe()
features = table.loc[:, result.feature_names]
display(features)
display(result.processing)
print("Run summary:", result.summary)
print("Saved results:", result.output_dir)

EEG amplitudes are reported in **µV**, and latencies in **ms**. For example, `ERP__target__late_positive__peak_amplitude_uV__Pz` is the target-condition peak amplitude at Pz.

These checks confirm that the complete local workflow recovers the synthetic signal. They do not validate cleaning performance on real EEG.

In [ ]:
amplitude_col = "ERP__target__late_positive__peak_amplitude_uV__Pz"
latency_col = "ERP__target__late_positive__peak_latency_ms__Pz"
assert features.shape == (len(participant_ids), 14)
assert (features[amplitude_col] - 6.0).abs().lt(0.2).all()
assert (features[latency_col] - 300.0).abs().le(12.0).all()
print("Synthetic amplitude and latency checks passed.")

### Read the quality-control reports

`success` means all requested IDPs are available; `partial` means some are available and others are `NaN`. Both are included in the cohort table. Participants with no usable IDPs, or processing errors, are reported as failed. If every participant fails, the pipeline raises an error.

QC explains missing measurements, retained trial counts and rejected trials. A `boundary_peak` flag means the selected peak falls at the edge of its window; review the waveform and window before interpreting it. QC is not included in `feature_names`.

Results are saved automatically in a new `results/erp/<run-id>/` folder. Existing runs are retained. Each participant also has a processing log and preprocessing report.

In [ ]:
qc = pd.read_csv(result.paths["qc"])
display(qc)
display(qc[["condition", "n_events", "n_trials", "n_rejected"]].drop_duplicates())

participant_dir = result.output_dir / "participants" / participant_ids[0]
preprocessing_qc = json.loads((participant_dir / "preprocessing.json").read_text())
print("ICA mode:", preprocessing_qc["ica_mode_effective"])
print("Excluded ICA components:", preprocessing_qc["ica_excluded"])
assert qc["status"].eq("ok").all()

## 6. Use your own EEG recordings

Change the dataset location, task text, file extension, event mapping and component definitions to match your study. Do **not** reuse the synthetic preprocessing settings above for raw EEG.

The following example is a template; it is not executed by Run All. It reuses the component definitions above, so adjust their conditions, channels and windows before running it.

```python
study_dataset = Dataset(
    name="my_oddball_study",
    root="/data/oddball",  # contains participant folders
    task="oddball", extension=".vhdr",
    erp_events=ERPEventSpec(
        mode="annotations",
        annotation_map={"standard": "Stimulus/S  5", "target": "Stimulus/S  6"},
    ),
)

study_config = Config(
    analysis_type="erp", which_sensor="eeg",
    erp=ERPSettings(
        components=erp.components,
        preprocessing=ERPPreprocessing(
            tmin=-0.2, tmax=0.8, baseline=(-0.2, 0.0),
            reference="average", l_freq=0.1, h_freq=35.0,
            ica_mode="auto", ica_l_freq=1.0,
            ica_corr_threshold=0.8, ica_label_threshold=0.9,
            min_trials=10, reject={"eeg": 150e-6},
        ),
    ),
)

study_result = Pipeline(
    config=study_config, output_dir="results", on_error="continue",
).run(study_dataset)
```

These are starting settings to assess for your protocol, not validated defaults for every ERP. `auto` uses usable EOG/ECG channels, otherwise ICLabel. ICLabel requires average reference and valid EEG sensor locations; its training used 1–100 Hz data, so a different fit filter is recorded as a QC warning. Explicit `physiological`, `iclabel`, and `none` modes are also available. `none` disables ICA, not all preprocessing.

Ensure the recording has correct EEG/EOG/ECG channel types, bad-channel markings and sensor positions. The local worker also reads matching channel tables and supported position sidecars. If positions are missing, configure a compatible template as shown below. Keep outputs outside the input dataset folder.

For annotations, inspect the descriptions in a representative recording before choosing the map:

```python
example_raw = mne.io.read_raw_brainvision("/data/oddball/sub-001/sub-001_oddball_eeg.vhdr")
print(sorted(set(example_raw.annotations.description)))
```

For events in a stimulus channel, replace `erp_events` with:

```python
ERPEventSpec(
    mode="stim", stim_channel="STI 014",
    event_id={"standard": 5, "target": 6},
)
```

Use your actual channel name and trigger codes. Do not pass an annotation map in stimulus mode. External BIDS `events.tsv` files are not read automatically.

To combine datasets with different trigger encodings, define a `Dataset` for each and call `Pipeline(...).run([dataset_a, dataset_b, ...])`. Use consistent logical condition names and unique participant IDs across datasets. Optional demographics can be supplied through `Dataset(demographics=..., participant_id=...)`; select `result.feature_names` when you need only ERP responses.

### Electrode positions

Valid embedded positions take priority, followed by adjacent BIDS electrode
files and then montage CSV files. If positions are absent, select a template
once in `Dataset`; no preparation script or generated sidecars are needed:

```python
# Add this argument to Dataset(...).
erp_montage={
    "template": "auto",  # 1005 template; does not infer the actual cap geometry.
    "exclude_channels": ["O9", "O10"],  # Only if these should be excluded.
}
```

You can select an MNE built-in template name instead of `auto`. Omit this
argument to use supplied positions only. Templates are approximations, not
participant measurements. Ear labels `TP9 LEFT EAR` and `TP10 RIGHT EAR` map
to TP9 and TP10 while preserving the original recording names. Unmatched
channels produce an error; explicit exclusions cannot include requested IDP
channels. The input files remain untouched.

Template use is announced and recorded in `processing.log`, `montage.json`,
`preprocessing.json`, and `provenance.json`. If an existing CSV came from a
template, add `"supplied_kind": "template"`; use `"measured"` only for known
measured positions. Otherwise, the origin of supplied positions is recorded
as unknown.

Adjacent BIDS `*_electrodes.tsv` and matching `*_coordsystem.json` support
**CapTrak head coordinates** in `m`, `cm`, or `mm`. Other coordinate systems
require an explicit transformation before loading; full BIDS inheritance is
not implemented. CSV files require `Channel,X,Y,Z` in head-frame metres.
A recording-specific CSV takes priority over a single unambiguous local CSV.
Incomplete supplied positions are never mixed with template coordinates.


### Optional AutoReject epoch cleaning

The default `rejection_mode="threshold"` uses the fixed amplitude cutoff. To enable learned thresholds and channel interpolation for your real recordings, add `rejection_mode="autoreject"` to `ERPSettings(...)`. You can also supply `autoreject={"cv": 5}` to choose the cross-validation folds.

AutoReject replaces `preprocessing.reject`; BAD annotations, recording edges and any `preprocessing.flat` threshold still apply. It fits each participant's conditions together without changing epoch times or baseline. It needs at least four usable EEG channels with valid distinct locations and `max(5, cv)` retained epochs. These are computational minimums, not scientific sufficiency criteria. **The two-channel synthetic example above cannot use AutoReject.**

Inspect `n_autoreject_rejected` and `n_interpolated_trials` in the QC table, and `epoch_rejection` in the saved preprocessing report for learned thresholds and per-trial repair decisions. Interpolation counts cover retained trials only. Unmet prerequisites cause a participant failure; there is no silent fallback to threshold cleaning. Validate the effect on your real EEG responses before interpreting IDPs.


## 7. Optional: inspect waveforms with the standalone API

The cohort API returns tables and saved reports. When you also need in-memory epochs or evoked responses, use the standalone API for a selected recording. This example reruns the first synthetic participant using exactly the same configuration. It is optional for cohort extraction.

In [ ]:
from meganorm.src.erp import extract_recording, compute_evokeds, extract_erp_idps
from meganorm.plots.plot_erp import plot_erp
import matplotlib.pyplot as plt

runtime_config, components = config.erp.to_runtime()
participant_id = participant_ids[0]
recording = data_root / participant_id / f"{participant_id}_oddball_raw.fif"
recording_result = extract_recording(
    recording, participant_id, components,
    config=runtime_config, event_spec=dataset.erp_events,
)
fig, axes = plot_erp(recording_result.evokeds, components=components, channels=("Cz", "Pz"))
plt.show()
np.testing.assert_allclose(
    recording_result.features.loc[participant_id], features.loc[participant_id],
)

### Already-prepared epochs or evoked responses

If you already have cleaned, event-locked epochs, average them with `compute_evokeds` and extract measurements directly. This bypasses raw preprocessing. Use ordinary condition averages with accurate `nave` trial counts, not weighted contrasts.

In [ ]:
prepared_evokeds = compute_evokeds(recording_result.epochs)
prepared_features, prepared_qc = extract_erp_idps(
    participant_id, prepared_evokeds, components, min_trials=10,
)
pd.testing.assert_frame_equal(prepared_features, recording_result.features)

## Current scope

The local raw-data ERP workflow supports EEG, annotation/stimulus events, threshold rejection or AutoReject, and sensor-level measurements. Automatic continuous bad-channel detection, sensor-group averages, condition contrasts, source reconstruction and ERP SLURM execution are not yet supported. Multiple recording candidates are not automatically combined into sessions.

Prepared-evoked extraction can also measure MEG using `ERPComponent(sensor_type="mag")` or `"grad"` with matching sensors; amplitudes are then in T or T/m. This does not enable raw MEG ERP preprocessing in the local cohort API.

Review processing outcomes, trial counts, waveforms and preprocessing QC before using ERP IDPs in downstream analyses. Validate your study protocol on representative real recordings.

See the [ERP quickstart](../erp_quickstart.md) for a compact real-data example.